In [1]:
!pip install -q groq pdfplumber pymupdf pandas openpyxl tqdm pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 597.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.6 MB/s eta 0:00:00


In [11]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Clé trouvée:", key is not None)
print("Début clé:", key[:4] if key else None)
print("Longueur:", len(key) if key else 0)

Clé trouvée: True
Début clé: gsk_
Longueur: 56


In [12]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Clé trouvée:", key is not None)
print("Début clé:", key[:4])

if key.startswith("xai-"):
    raise ValueError("❌ Ceci est une clé xAI/Grok, pas Groq.")

if not key.startswith("gsk_"):
    print("⚠️ Vérifie la clé. Une clé Groq commence souvent par gsk_.")

print("✅ Clé Groq correcte")

Clé trouvée: True
Début clé: gsk_
✅ Clé Groq correcte


In [13]:
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Réponds seulement: OK"}
    ],
    temperature=0
)

print(response.choices[0].message.content)

OK


In [14]:
!pip install -q requests

In [16]:
import requests
from google.colab import userdata

XAI_API_KEY = userdata.get("GROQ_API_KEY")

if not XAI_API_KEY:
    raise ValueError("❌ XAI_API_KEY introuvable dans Colab Secrets.")

print("Début clé:", XAI_API_KEY[:4])

url = "https://api.x.ai/v1/chat/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {XAI_API_KEY}"
}

payload = {
    "model": "grok-4.3",
    "messages": [
        {"role": "user", "content": "Réponds seulement: OK"}
    ],
    "temperature": 0
}

response = requests.post(url, headers=headers, json=payload)

print("Status:", response.status_code)
print(response.text)

Début clé: gsk_
Status: 400
{"code":"Client specified an invalid argument","error":"Incorrect API key provided: gs***GU. You can obtain an API key from https://console.x.ai."}


In [18]:
import os
import re
import io
import json
import time
import base64
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import pdfplumber
import fitz  # PyMuPDF
from PIL import Image
from tqdm import tqdm
from groq import Groq
from google.colab import userdata

# ═══════════════════════════════════════════════════════════════════════
# CONFIG PROJECT PI
# ═══════════════════════════════════════════════════════════════════════

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_AI manquante. Ajoute-la dans Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

BASE_DIR = Path("/content/drive/MyDrive/medical_project")

INPUT_DIR = BASE_DIR / "input_rapports"
OUTPUT_DIR = BASE_DIR / "groq_pi_outputs"
OUTPUT_JSON_DIR = OUTPUT_DIR / "json"
OUTPUT_EXCEL = OUTPUT_DIR / "resultats_project_pi_groq.xlsx"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

# Modèle texte pour extraction JSON finale
TEXT_MODEL = "openai/gpt-oss-20b"

# Modèle vision pour PDF scanné ou image
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

MIN_TEXT_CHARS = 150

print("✅ Configuration Groq prête")
print("Input folder :", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Configuration Groq prête
Input folder : /content/drive/MyDrive/medical_project/input_rapports
Output folder: /content/drive/MyDrive/medical_project/groq_pi_outputs


In [19]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()

    return text


def anonymize_text(text: str) -> str:
    """
    Sécurité simple avant envoi API.
    Important si tes rapports contiennent patient, téléphone, email, etc.
    """
    if not text:
        return ""

    # Emails
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL]",
        text
    )

    # Téléphones
    text = re.sub(
        r"(\+?\d[\d\s\-.]{7,}\d)",
        "[TELEPHONE]",
        text
    )

    # Numéros longs
    text = re.sub(
        r"\b\d{8,}\b",
        "[ID_NUMERIQUE]",
        text
    )

    # Patient / patiente
    text = re.sub(
        r"(?i)\b(patient|patiente)\s*[:\-]?\s+[A-ZÉÈÀÂÊÎÔÛÇ][\wÉÈÀÂÊÎÔÛÇéèàâêîôûç'\- ]+",
        "patient: [PATIENT]",
        text
    )

    return clean_text(text)

In [20]:
def extract_text_with_pdfplumber(pdf_path: Path) -> str:
    pages_text = []

    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text() or ""
                page_text = clean_text(page_text)

                if page_text:
                    pages_text.append(f"\n--- PAGE {page_number} ---\n{page_text}")

    except Exception as e:
        print(f"⚠️ pdfplumber erreur pour {pdf_path.name}: {e}")
        return ""

    return clean_text("\n".join(pages_text))

In [21]:
def image_to_base64_jpeg(image: Image.Image, max_size_mb: float = 3.5) -> str:
    """
    Convertit une image PIL en base64 JPEG.
    On compresse pour rester sous la limite base64 Groq.
    """

    image = image.convert("RGB")

    quality = 90

    while quality >= 40:
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=quality, optimize=True)
        size_mb = len(buffer.getvalue()) / (1024 * 1024)

        if size_mb <= max_size_mb:
            return base64.b64encode(buffer.getvalue()).decode("utf-8")

        quality -= 10

    # Dernier fallback : resize
    width, height = image.size
    image = image.resize((int(width * 0.75), int(height * 0.75)))

    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=60, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def pdf_pages_to_images(pdf_path: Path, max_pages: int = 10, zoom: float = 2.0) -> List[Image.Image]:
    """
    Convertit les pages PDF en images avec PyMuPDF.
    max_pages évite d'envoyer un énorme PDF gratuitement.
    """

    images = []

    doc = fitz.open(str(pdf_path))

    total_pages = min(len(doc), max_pages)

    for page_index in range(total_pages):
        page = doc[page_index]

        matrix = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=matrix, alpha=False)

        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)

    doc.close()

    return images

In [22]:
VISION_SYSTEM_PROMPT = """
Tu es un assistant de lecture de document pour Project PI.

Tu reçois une image d'une page de rapport médical/commercial.
Lis tout le texte visible dans l'image.

Règles:
- Retourne seulement le texte lu.
- Garde les phrases complètes.
- Ne résume pas.
- Ne crée pas de JSON ici.
- Si une partie est illisible, écris [ILLISIBLE].
"""


def extract_text_from_image_with_groq(image: Image.Image, page_number: int) -> str:
    image_b64 = image_to_base64_jpeg(image)

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "system",
                "content": VISION_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"Lis le texte de cette page {page_number} du rapport."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_b64}"
                        }
                    }
                ]
            }
        ],
        temperature=0,
        max_completion_tokens=2048
    )

    text = response.choices[0].message.content
    return clean_text(text)


def extract_scanned_pdf_text_with_groq_vision(pdf_path: Path, max_pages: int = 10) -> str:
    print("📸 PDF scanné détecté → conversion pages en images → Groq Vision")

    images = pdf_pages_to_images(pdf_path, max_pages=max_pages)

    all_text = []

    for i, image in enumerate(images, start=1):
        print(f"   → Lecture page scannée {i}/{len(images)} avec Groq Vision")
        page_text = extract_text_from_image_with_groq(image, i)

        if page_text:
            all_text.append(f"\n--- PAGE SCANNÉE {i} ---\n{page_text}")

        time.sleep(1)

    return clean_text("\n".join(all_text))

In [23]:
def extract_image_file_text_with_groq(image_path: Path) -> str:
    image = Image.open(image_path)
    return extract_text_from_image_with_groq(image, page_number=1)

In [24]:
SUPPORTED_EXTENSIONS = [".pdf", ".png", ".jpg", ".jpeg", ".txt", ".md"]


def extract_text_from_file(file_path: Path) -> Dict[str, Any]:
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        text = extract_text_with_pdfplumber(file_path)

        if len(text) >= MIN_TEXT_CHARS:
            return {
                "text": text,
                "mode": "pdfplumber_text_pdf"
            }

        # Pas OCR local : on envoie les pages comme images à Groq Vision
        text = extract_scanned_pdf_text_with_groq_vision(file_path)

        return {
            "text": text,
            "mode": "groq_vision_scanned_pdf"
        }

    if suffix in [".png", ".jpg", ".jpeg"]:
        text = extract_image_file_text_with_groq(file_path)

        return {
            "text": text,
            "mode": "groq_vision_image"
        }

    if suffix in [".txt", ".md"]:
        try:
            text = file_path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            text = file_path.read_text(encoding="latin-1")

        return {
            "text": clean_text(text),
            "mode": "text_file"
        }

    raise ValueError(f"Extension non supportée: {suffix}")

In [25]:
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "nom_medecin": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "specialite_medecin": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "medicaments": {
            "type": "array",
            "items": {"type": "string"}
        },
        "gadgets": {
            "type": "array",
            "items": {"type": "string"}
        },
        "type_visite": {
            "type": "string",
            "enum": [
                "prospection",
                "suivi",
                "lancement",
                "formation",
                "réclamation",
                "autre",
                "inconnu"
            ]
        },
        "objectif_visite": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "reponse_medecin": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "prochaine_action": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "commentaire_visite": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "niveau_interet": {
            "type": "string",
            "enum": [
                "faible",
                "moyen",
                "élevé",
                "très élevé",
                "inconnu"
            ]
        },
        "resume_rapport": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        },
        "champs_manquants": {
            "type": "array",
            "items": {"type": "string"}
        },
        "alertes_qualite": {
            "type": "array",
            "items": {"type": "string"}
        },
        "confidence_champs": {
            "type": "object",
            "properties": {
                "nom_medecin": {"type": "integer"},
                "specialite_medecin": {"type": "integer"},
                "medicaments": {"type": "integer"},
                "gadgets": {"type": "integer"},
                "type_visite": {"type": "integer"},
                "objectif_visite": {"type": "integer"},
                "reponse_medecin": {"type": "integer"},
                "prochaine_action": {"type": "integer"},
                "commentaire_visite": {"type": "integer"},
                "niveau_interet": {"type": "integer"}
            },
            "required": [
                "nom_medecin",
                "specialite_medecin",
                "medicaments",
                "gadgets",
                "type_visite",
                "objectif_visite",
                "reponse_medecin",
                "prochaine_action",
                "commentaire_visite",
                "niveau_interet"
            ],
            "additionalProperties": False
        },
        "confidence_global": {
            "type": "integer"
        },
        "extrait_source_utilise": {
            "anyOf": [{"type": "string"}, {"type": "null"}]
        }
    },
    "required": [
        "nom_medecin",
        "specialite_medecin",
        "medicaments",
        "gadgets",
        "type_visite",
        "objectif_visite",
        "reponse_medecin",
        "prochaine_action",
        "commentaire_visite",
        "niveau_interet",
        "resume_rapport",
        "champs_manquants",
        "alertes_qualite",
        "confidence_champs",
        "confidence_global",
        "extrait_source_utilise"
    ],
    "additionalProperties": False
}

In [26]:
SYSTEM_EXTRACTION_PROMPT = """
Tu es un système d'extraction d'information pour Project PI.

Tu reçois le texte d'un rapport de visite médicale/commerciale.
Tu dois extraire les champs nécessaires pour construire un rapport final.

Règles:
1. Ne jamais inventer une information absente.
2. Si une information est absente, utiliser null, "inconnu", ou liste vide.
3. Garder les phrases importantes comme phrases complètes.
4. Ne pas retourner un seul mot si le champ est normalement une phrase.
5. Extraire les médicaments dans une liste.
6. Extraire les gadgets, supports, brochures, échantillons, outils dans une liste.
7. type_visite doit être: prospection, suivi, lancement, formation, réclamation, autre, inconnu.
8. niveau_interet doit être: faible, moyen, élevé, très élevé, inconnu.
9. Ajouter champs_manquants.
10. Ajouter alertes_qualite si le texte est court, incomplet, bruité ou contradictoire.
11. Les confidence_champs doivent être entre 0 et 100.
12. confidence_global doit être entre 0 et 100.
13. Retourne uniquement le JSON demandé.
"""


def build_extraction_prompt(text: str) -> str:
    return f"""
Voici le texte extrait du rapport:

{text}

Extrais les informations nécessaires pour Project PI.
Retourne uniquement le JSON final.
"""

In [33]:
# ═══════════════════════════════════════════════════════════════════════
# CELLULE 10 CORRIGÉE — Appel Groq robuste JSON
# ═══════════════════════════════════════════════════════════════════════

def clamp_int(value, min_value=0, max_value=100):
    try:
        value = int(value)
    except Exception:
        value = 0

    return max(min_value, min(max_value, value))


def normalize_extraction_json(data: Dict[str, Any], source_text: str = "") -> Dict[str, Any]:
    """
    Cette fonction répare le JSON si Groq oublie un champ.
    Elle évite que tout le batch s'arrête.
    """

    allowed_type_visite = [
        "prospection",
        "suivi",
        "lancement",
        "formation",
        "réclamation",
        "autre",
        "inconnu"
    ]

    allowed_niveau_interet = [
        "faible",
        "moyen",
        "élevé",
        "très élevé",
        "inconnu"
    ]

    required_fields_defaults = {
        "nom_medecin": None,
        "specialite_medecin": None,
        "medicaments": [],
        "gadgets": [],
        "type_visite": "inconnu",
        "objectif_visite": None,
        "reponse_medecin": None,
        "prochaine_action": None,
        "commentaire_visite": None,
        "niveau_interet": "inconnu",
        "resume_rapport": None,
        "champs_manquants": [],
        "alertes_qualite": [],
        "confidence_champs": {},
        "confidence_global": 0,
        "extrait_source_utilise": None,
    }

    # Ajouter les champs manquants
    for field, default_value in required_fields_defaults.items():
        if field not in data:
            data[field] = default_value

    # Corriger les listes
    for list_field in ["medicaments", "gadgets", "champs_manquants", "alertes_qualite"]:
        if data.get(list_field) is None:
            data[list_field] = []

        if isinstance(data[list_field], str):
            data[list_field] = [data[list_field]]

        if not isinstance(data[list_field], list):
            data[list_field] = []

        data[list_field] = [
            str(x).strip()
            for x in data[list_field]
            if str(x).strip()
        ]

    # Corriger enums
    if data.get("type_visite") not in allowed_type_visite:
        data["type_visite"] = "inconnu"

    if data.get("niveau_interet") not in allowed_niveau_interet:
        data["niveau_interet"] = "inconnu"

    # Corriger confidence_champs
    default_confidence_champs = {
        "nom_medecin": 0,
        "specialite_medecin": 0,
        "medicaments": 0,
        "gadgets": 0,
        "type_visite": 0,
        "objectif_visite": 0,
        "reponse_medecin": 0,
        "prochaine_action": 0,
        "commentaire_visite": 0,
        "niveau_interet": 0,
    }

    if not isinstance(data.get("confidence_champs"), dict):
        data["confidence_champs"] = {}

    for field in default_confidence_champs:
        data["confidence_champs"][field] = clamp_int(
            data["confidence_champs"].get(field, 0)
        )

    data["confidence_global"] = clamp_int(data.get("confidence_global", 0))

    # Si Groq a oublié extrait_source_utilise
    if not data.get("extrait_source_utilise"):
        data["extrait_source_utilise"] = source_text[:400] if source_text else None

    # Ajouter automatiquement champs manquants si vide
    important_fields = [
        "nom_medecin",
        "specialite_medecin",
        "medicaments",
        "type_visite",
        "objectif_visite",
        "reponse_medecin",
        "prochaine_action",
        "niveau_interet",
    ]

    detected_missing = []

    for field in important_fields:
        value = data.get(field)

        if value is None:
            detected_missing.append(field)

        elif isinstance(value, str) and value.strip().lower() in ["", "inconnu", "non mentionné"]:
            detected_missing.append(field)

        elif isinstance(value, list) and len(value) == 0:
            detected_missing.append(field)

    existing_missing = set(data.get("champs_manquants", []))
    for field in detected_missing:
        existing_missing.add(field)

    data["champs_manquants"] = sorted(list(existing_missing))

    return data


def extract_json_from_text(text: str) -> Dict[str, Any]:
    """
    Au cas où le modèle retourne du texte autour du JSON.
    """
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        json_text = text[start:end + 1]
        return json.loads(json_text)

    raise ValueError("Aucun JSON valide trouvé dans la réponse Groq.")


def call_groq_structured_extraction(text: str, max_retries: int = 3) -> Dict[str, Any]:
    """
    Version robuste:
    1. Essaye json_schema strict avec beaucoup de tokens.
    2. Si Groq échoue, utilise json_object.
    3. Normalise localement les champs manquants.
    """

    safe_text = anonymize_text(clean_text(text))

    if len(safe_text) < 30:
        raise ValueError("Texte trop court pour extraction.")

    last_error = None

    # Tentative 1 : JSON Schema strict
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=TEXT_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_EXTRACTION_PROMPT
                    },
                    {
                        "role": "user",
                        "content": build_extraction_prompt(safe_text)
                    }
                ],
                temperature=0,
                reasoning_effort="low",
                max_completion_tokens=4096,
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "project_pi_report_extraction",
                        "strict": True,
                        "schema": REPORT_SCHEMA
                    }
                }
            )

            content = response.choices[0].message.content
            data = json.loads(content)
            data = normalize_extraction_json(data, source_text=safe_text)

            return data

        except Exception as e:
            last_error = e
            print(f"⚠️ Strict JSON tentative {attempt + 1}/{max_retries} échouée: {e}")
            time.sleep(2 ** attempt)

    print("⚠️ Passage au fallback json_object...")

    # Tentative 2 : JSON Object fallback
    fallback_prompt = f"""
Tu dois retourner uniquement un objet JSON valide.

Le JSON doit contenir exactement ces champs:
- nom_medecin
- specialite_medecin
- medicaments
- gadgets
- type_visite
- objectif_visite
- reponse_medecin
- prochaine_action
- commentaire_visite
- niveau_interet
- resume_rapport
- champs_manquants
- alertes_qualite
- confidence_champs
- confidence_global
- extrait_source_utilise

Règles:
- Ne retourne aucun texte avant ou après le JSON.
- Si une information est absente, utilise null, [] ou "inconnu".
- confidence_global doit être un nombre entre 0 et 100.
- extrait_source_utilise doit être une courte phrase source du rapport.
- confidence_champs doit contenir les scores de tous les champs.

Texte du rapport:
{safe_text}
"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=TEXT_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "Tu es un extracteur JSON pour Project PI. Réponds uniquement avec JSON valide."
                    },
                    {
                        "role": "user",
                        "content": fallback_prompt
                    }
                ],
                temperature=0,
                reasoning_effort="low",
                max_completion_tokens=4096,
                response_format={
                    "type": "json_object"
                }
            )

            content = response.choices[0].message.content
            data = extract_json_from_text(content)
            data = normalize_extraction_json(data, source_text=safe_text)

            return data

        except Exception as e:
            last_error = e
            print(f"⚠️ Fallback JSON tentative {attempt + 1}/{max_retries} échouée: {e}")
            time.sleep(2 ** attempt)

    raise RuntimeError(f"❌ Groq extraction failed après fallback: {last_error}")

In [34]:
FIELD_WEIGHTS = {
    "nom_medecin": 10,
    "specialite_medecin": 8,
    "medicaments": 10,
    "gadgets": 5,
    "type_visite": 10,
    "objectif_visite": 15,
    "reponse_medecin": 15,
    "prochaine_action": 15,
    "commentaire_visite": 5,
    "niveau_interet": 7,
}


def is_filled(value: Any) -> bool:
    if value is None:
        return False

    if isinstance(value, str):
        value = value.strip().lower()
        return value not in [
            "",
            "null",
            "none",
            "inconnu",
            "non mentionné",
            "non precise",
            "non précisé"
        ]

    if isinstance(value, list):
        return len([x for x in value if str(x).strip()]) > 0

    return True


def calculate_report_score(data: Dict[str, Any]) -> Dict[str, Any]:
    total_weight = sum(FIELD_WEIGHTS.values())
    filled_score = 0
    missing_fields = []

    for field, weight in FIELD_WEIGHTS.items():
        if is_filled(data.get(field)):
            filled_score += weight
        else:
            missing_fields.append(field)

    score_completion = round((filled_score / total_weight) * 100)

    confidence_champs = data.get("confidence_champs", {})
    confidence_values = []

    for value in confidence_champs.values():
        try:
            value = int(value)
            value = max(0, min(100, value))
            confidence_values.append(value)
        except Exception:
            pass

    if confidence_values:
        confidence_moyenne = round(sum(confidence_values) / len(confidence_values))
    else:
        confidence_moyenne = 0

    confidence_global = int(data.get("confidence_global") or 0)
    confidence_global = max(0, min(100, confidence_global))

    score_rapport = round(
        0.55 * score_completion
        + 0.30 * confidence_moyenne
        + 0.15 * confidence_global
    )

    if score_rapport >= 85:
        qualite = "excellent"
    elif score_rapport >= 70:
        qualite = "bon"
    elif score_rapport >= 50:
        qualite = "moyen"
    else:
        qualite = "faible"

    return {
        "score_rapport": score_rapport,
        "score_completion": score_completion,
        "confidence_moyenne_champs": confidence_moyenne,
        "confidence_global_code": confidence_global,
        "qualite_rapport": qualite,
        "champs_vides_detectes_code": missing_fields
    }

In [35]:
def build_final_report(data: Dict[str, Any], score_info: Dict[str, Any]) -> str:
    medicaments = data.get("medicaments") or []
    gadgets = data.get("gadgets") or []

    meds = ", ".join(medicaments) if medicaments else "Non mentionné"
    gads = ", ".join(gadgets) if gadgets else "Non mentionné"

    champs_manquants = data.get("champs_manquants") or []
    alertes_qualite = data.get("alertes_qualite") or []

    rapport = f"""
RAPPORT DE VISITE

Médecin:
- Nom: {data.get("nom_medecin") or "Non mentionné"}
- Spécialité: {data.get("specialite_medecin") or "Non mentionné"}

Produits:
- Médicaments: {meds}
- Gadgets / supports: {gads}

Visite:
- Type de visite: {data.get("type_visite") or "Non mentionné"}
- Objectif: {data.get("objectif_visite") or "Non mentionné"}

Réponse du médecin:
{data.get("reponse_medecin") or "Non mentionné"}

Prochaine action:
{data.get("prochaine_action") or "Non mentionné"}

Commentaire:
{data.get("commentaire_visite") or "Non mentionné"}

Niveau d'intérêt:
{data.get("niveau_interet") or "Non mentionné"}

Résumé:
{data.get("resume_rapport") or "Non mentionné"}

Qualité:
- Score rapport: {score_info["score_rapport"]}/100
- Qualité: {score_info["qualite_rapport"]}
- Score complétion: {score_info["score_completion"]}/100
- Confiance moyenne champs: {score_info["confidence_moyenne_champs"]}/100

Champs manquants:
{", ".join(champs_manquants) if champs_manquants else "Aucun"}

Alertes qualité:
{", ".join(alertes_qualite) if alertes_qualite else "Aucune"}
""".strip()

    return rapport

In [36]:
def process_one_file(file_path: Path) -> Dict[str, Any]:
    print("\n" + "=" * 80)
    print(f"📄 Traitement: {file_path.name}")
    print("=" * 80)

    if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Extension non supportée: {file_path.suffix}")

    extracted = extract_text_from_file(file_path)

    text = extracted["text"]
    extraction_mode = extracted["mode"]

    print(f"Mode extraction: {extraction_mode}")
    print(f"Caractères extraits: {len(text)}")

    if len(text) < 30:
        raise ValueError("Texte extrait trop court. Le fichier est peut-être illisible.")

    data = call_groq_structured_extraction(text)

    score_info = calculate_report_score(data)

    rapport_final = build_final_report(data, score_info)

    output = {
        "file_name": file_path.name,
        "file_path": str(file_path),
        "extraction_mode": extraction_mode,
        "raw_text_chars": len(text),
        **data,
        **score_info,
        "rapport_final": rapport_final
    }

    json_path = OUTPUT_JSON_DIR / f"{file_path.stem}_project_pi_groq.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"✅ JSON sauvegardé: {json_path}")
    print(f"✅ Score rapport: {score_info['score_rapport']}/100")
    print(f"✅ Qualité: {score_info['qualite_rapport']}")

    return output

In [37]:
files = []

for ext in SUPPORTED_EXTENSIONS:
    files.extend(INPUT_DIR.glob(f"*{ext}"))

files = sorted(files)

if not files:
    raise FileNotFoundError(f"Aucun fichier trouvé dans {INPUT_DIR}")

test_file = files[0]

result = process_one_file(test_file)

print("\n" + "=" * 80)
print(result["rapport_final"])
print("=" * 80)


📄 Traitement: TEST.png
Mode extraction: groq_vision_image
Caractères extraits: 1420
✅ JSON sauvegardé: /content/drive/MyDrive/medical_project/groq_pi_outputs/json/TEST_project_pi_groq.json
✅ Score rapport: 100/100
✅ Qualité: excellent

RAPPORT DE VISITE

Médecin:
- Nom: Dr Leila Mnif
- Spécialité: cardiologie

Produits:
- Médicaments: Cardiolis
- Gadgets / supports: 10 échantillons, porte-documents promotionnel

Visite:
- Type de visite: lancement
- Objectif: présenter Cardiolis, expliquer son intérêt thérapeutique et initier son référencement dans les prescriptions des patients à risque cardiovasculaire modéré

Réponse du médecin:
la proposition était pertinente, le positionnement du produit semblait cohérent, mais elle souhaite comparer davantage les données avant de modifier ses habitudes de prescription

Prochaine action:
revenir avec des données comparatives détaillées et discuter d'une première série de prescriptions ciblées

Commentaire:
accueil sérieux, intérêt clinique réel e

In [38]:
def process_folder(input_dir: Path = INPUT_DIR) -> pd.DataFrame:
    files = []

    for ext in SUPPORTED_EXTENSIONS:
        files.extend(input_dir.glob(f"*{ext}"))

    files = sorted(files)

    if not files:
        raise FileNotFoundError(f"Aucun fichier trouvé dans {input_dir}")

    print(f"🗂️ {len(files)} fichier(s) trouvé(s)")

    results = []
    errors = []

    for file_path in tqdm(files):
        try:
            output = process_one_file(file_path)
            results.append(output)

        except Exception as e:
            print(f"❌ Erreur avec {file_path.name}: {e}")
            errors.append({
                "file_name": file_path.name,
                "error": str(e)
            })

    df = pd.DataFrame(results)

    if len(df) > 0:
        df.to_excel(OUTPUT_EXCEL, index=False)
        print(f"\n✅ Excel sauvegardé: {OUTPUT_EXCEL}")

    if errors:
        error_path = OUTPUT_DIR / "errors.json"

        with open(error_path, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)

        print(f"⚠️ Erreurs sauvegardées: {error_path}")

    return df


df_results = process_folder(INPUT_DIR)
df_results.head()

🗂️ 7 fichier(s) trouvé(s)


  0%|          | 0/7 [00:00<?, ?it/s]


📄 Traitement: TEST.png
Mode extraction: groq_vision_image
Caractères extraits: 1420


 14%|█▍        | 1/7 [00:06<00:39,  6.66s/it]

✅ JSON sauvegardé: /content/drive/MyDrive/medical_project/groq_pi_outputs/json/TEST_project_pi_groq.json
✅ Score rapport: 100/100
✅ Qualité: excellent

📄 Traitement: test3.pdf
Mode extraction: pdfplumber_text_pdf
Caractères extraits: 1435


...

 86%|████████▌ | 6/7 [01:53<00:24, 24.24s/it]

✅ JSON sauvegardé: /content/drive/MyDrive/medical_project/groq_pi_outputs/json/text1_project_pi_groq.json
✅ Score rapport: 100/100
✅ Qualité: excellent

📄 Traitement: text2.pdf
Mode extraction: pdfplumber_text_pdf
Caractères extraits: 1460


100%|██████████| 7/7 [01:54<00:00, 16.33s/it]

✅ JSON sauvegardé: /content/drive/MyDrive/medical_project/groq_pi_outputs/json/text2_project_pi_groq.json
✅ Score rapport: 100/100
✅ Qualité: excellent

✅ Excel sauvegardé: /content/drive/MyDrive/medical_project/groq_pi_outputs/resultats_project_pi_groq.xlsx


,file_name,file_path,extraction_mode,raw_text_chars,nom_medecin,specialite_medecin,medicaments,gadgets,type_visite,objectif_visite,...,confidence_champs,confidence_global,extrait_source_utilise,score_rapport,score_completion,confidence_moyenne_champs,confidence_global_code,qualite_rapport,champs_vides_detectes_code,rapport_final
0,TEST.png,/content/drive/MyDrive/medical_project/input_r...,groq_vision_image,1420,Dr Leila Mnif,cardiologie,[Cardiolis],"[10 échantillons, porte-documents promotionnel]",lancement,"présenter Cardiolis, expliquer son intérêt thé...",...,"{'nom_medecin': 100, 'specialite_medecin': 100...",100,"Le 16/04/2026, une visite de lancement a été r...",100,100,100,100,excellent,[],RAPPORT DE VISITE\n\nMédecin:\n- Nom: Dr Leila...
1,test3.pdf,/content/drive/MyDrive/medical_project/input_r...,pdfplumber_text_pdf,1435,Dr Leila Mnif,cardiologie,"[Cardiolis, Vascorel]","[porte-documents promotionnel, échantillons]",lancement,"présenter Cardiolis, expliquer son intérêt thé...",...,"{'nom_medecin': 95, 'specialite_medecin': 95, ...",90,"Le 16/04/2026, une visite de lancement a été r...",96,100,92,90,excellent,[],RAPPORT DE VISITE\n\nMédecin:\n- Nom: Dr Leila...
2,test_avec_tous_les_information.pdf,/content/drive/MyDrive/medical_project/input_r...,pdfplumber_text_pdf,1684,Dr Mariem Trabelsi,gynécologie,"[Ortho-Tri-Cyclen, Yasmin]",[stylo promotionnel],suivi,"Renforcer la prescription d’Ortho‑Tri‑Cyclen, ...",...,"{'nom_medecin': 100, 'specialite_medecin': 100...",100,"Le 15/04/2026, dans le cadre d’une visite de s...",100,100,100,100,excellent,[],RAPPORT DE VISITE\n\nMédecin:\n- Nom: Dr Marie...
3,test_moyenne.pdf,/content/drive/MyDrive/medical_project/input_r...,pdfplumber_text_pdf,1291,Dr Mariem Trabelsi,gynécologie,[Ortho-Tri-Cyclen],[stylo promotionnel],suivi,renforcer le positionnement du produit et répo...,...,"{'nom_medecin': 100, 'specialite_medecin': 100...",100,--- PAGE 1 ---\nLors d’une visite de suivi eff...,100,100,100,100,excellent,[],RAPPORT DE VISITE\n\nMédecin:\n- Nom: Dr Marie...
4,text1 (1).pdf,/content/drive/MyDrive/medical_project/input_r...,pdfplumber_text_pdf,1381,Dr Youssef Ben Amor,médecine générale,"[Respifast, Coughrelief]",[bloc-notes promotionnel],prospection,présenter le médicament Respifast et encourage...,...,"{'nom_medecin': 95, 'specialite_medecin': 95, ...",95,"Le 10/04/2026, dans le cadre d’une visite de p...",97,100,92,95,excellent,[],RAPPORT DE VISITE\n\nMédecin:\n- Nom: Dr Youss...


In [ ]:
from pathlib import Path
from datetime import datetime, date
import json
import re
from openpyxl import load_workbook
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.worksheet.datavalidation import DataValidation

TEMPLATE_FILENAME = "modele-rapport-delegue-medical (4).xlsx"
OUTPUT_TEMPLATE_XLSX_DIR = OUTPUT_DIR / "rapports_xlsx_template"


def resolve_template_path(template_path=None) -> Path:
    candidates = []

    if template_path:
        candidates.append(Path(template_path))

    candidates.extend([
        BASE_DIR / TEMPLATE_FILENAME,
        OUTPUT_DIR / TEMPLATE_FILENAME,
        INPUT_DIR / TEMPLATE_FILENAME,
        Path("/content") / TEMPLATE_FILENAME,
        Path.cwd() / TEMPLATE_FILENAME,
    ])

    candidates.extend(sorted(BASE_DIR.glob("modele-rapport-delegue-medical*.xlsx")))
    candidates.extend(sorted(Path("/content").glob("modele-rapport-delegue-medical*.xlsx")))
    candidates.extend(sorted(Path.cwd().glob("modele-rapport-delegue-medical*.xlsx")))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Template XLSX introuvable. Mets le fichier modele-rapport-delegue-medical (4).xlsx dans BASE_DIR, OUTPUT_DIR, INPUT_DIR ou /content."
    )


def get_by_path(data, key):
    current = data
    for part in key.split("."):
        if not isinstance(current, dict) or part not in current:
            return None
        current = current[part]
    return current


def first_value(data, keys, default=None):
    for key in keys:
        value = get_by_path(data, key) if "." in key else data.get(key)
        if value is None:
            continue
        if isinstance(value, str) and not value.strip():
            continue
        if isinstance(value, list) and len(value) == 0:
            continue
        return value
    return default


def clean_cell_value(value):
    if value is None:
        return None
    if isinstance(value, (datetime, date, int, float)):
        return value
    if isinstance(value, list):
        return "\n".join(str(x).strip() for x in value if str(x).strip()) or None
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    value = str(value).strip()
    return value if value else None


def parse_excel_date(value):
    if value is None or value == "":
        return None
    if isinstance(value, datetime):
        return value
    if isinstance(value, date):
        return value

    text = str(value).strip()
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y", "%Y-%m-%dT%H:%M:%S"]:
        try:
            return datetime.strptime(text[:19], fmt)
        except Exception:
            pass

    return text


def list_to_text(value):
    if value is None:
        return None
    if isinstance(value, list):
        return ", ".join(str(x).strip() for x in value if str(x).strip()) or None
    return clean_cell_value(value)


def write_cell(ws, cell, value):
    ws[cell] = clean_cell_value(value)


def normalize_products(data):
    products = first_value(data, [
        "produits",
        "produits_presentes",
        "produits_présentés",
        "medicaments_details",
        "médicaments_details"
    ], [])

    rows = []

    if isinstance(products, list) and products:
        for item in products:
            if isinstance(item, dict):
                name = first_value(item, ["nom", "nom_produit", "produit", "medicament", "médicament", "name"])
                rows.append({
                    "produit": name,
                    "commentaire": first_value(item, ["commentaire", "comment", "feedback", "promesse"]),
                    "opportunites": first_value(item, ["opportunites", "opportunités", "opportunite", "opportunité", "reponse_medecin", "réponse_médecin"]),
                    "benchmarking": first_value(item, ["benchmarking_concurrents", "benchmarking", "concurrents", "concurrent"]),
                    "echantillons": first_value(item, ["nombre_echantillons", "nombre_échantillons", "echantillons", "échantillons", "nb_echantillons", "samples"]),
                })
            else:
                rows.append({"produit": item})
    else:
        medicaments = first_value(data, ["medicaments", "médicaments", "medicament", "médicament"], [])
        if isinstance(medicaments, str):
            medicaments = [x.strip() for x in re.split(r"[,;\n]", medicaments) if x.strip()]
        for med in medicaments:
            rows.append({"produit": med})

    if rows:
        rows[0]["commentaire"] = rows[0].get("commentaire") or first_value(data, ["commentaire_visite", "commentaire", "resume_rapport"])
        rows[0]["opportunites"] = rows[0].get("opportunites") or first_value(data, ["opportunites", "opportunités", "reponse_medecin", "réponse_médecin"])
        rows[0]["benchmarking"] = rows[0].get("benchmarking") or first_value(data, ["benchmarking_concurrents", "benchmarking"])
        rows[0]["echantillons"] = rows[0].get("echantillons") or first_value(data, ["nombre_echantillons", "nombre_échantillons", "echantillons", "échantillons"])

    return rows[:6], rows[6:]


def update_medecin_sheet(wb, data):
    if "Médecin" not in wb.sheetnames:
        return

    ws = wb["Médecin"]
    name = clean_cell_value(first_value(data, ["nom_medecin", "nom_médecin", "nom_prospect", "medecin", "médecin"]))

    if not name:
        return

    row = None
    for r in range(2, max(ws.max_row, 2) + 1):
        if clean_cell_value(ws.cell(r, 1).value) == name:
            row = r
            break

    if row is None:
        row = max(ws.max_row + 1, 2)

    ws.cell(row, 1).value = name
    ws.cell(row, 2).value = clean_cell_value(first_value(data, ["secteur", "zone", "region", "région"]))
    ws.cell(row, 3).value = clean_cell_value(first_value(data, ["localite", "localité", "ville"] ))
    ws.cell(row, 4).value = clean_cell_value(first_value(data, ["specialite_medecin", "spécialité_médecin", "specialite", "spécialité"] ))
    ws.cell(row, 5).value = clean_cell_value(first_value(data, ["potentiel", "potential"] ))
    ws.cell(row, 6).value = clean_cell_value(first_value(data, ["telephone", "téléphone", "tel"] ))
    ws.cell(row, 7).value = clean_cell_value(first_value(data, ["gsm", "mobile"] ))
    ws.cell(row, 8).value = clean_cell_value(first_value(data, ["adresse", "address"] ))

    last_row = max(ws.max_row, row)

    if "Nom_du_prospect" in wb.defined_names:
        del wb.defined_names["Nom_du_prospect"]

    wb.defined_names.add(DefinedName("Nom_du_prospect", attr_text=f"'Médecin'!$A$2:$A${last_row}"))


def refill_report_validation(wb):
    if "Rapport" not in wb.sheetnames:
        return

    ws = wb["Rapport"]
    dv = DataValidation(type="list", formula1="=Nom_du_prospect", allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(ws["C2"])


def fill_template_workbook(wb, data):
    if "Rapport" not in wb.sheetnames:
        raise ValueError("La feuille 'Rapport' est introuvable dans le template.")

    ws = wb["Rapport"]

    write_cell(ws, "C2", first_value(data, ["nom_medecin", "nom_médecin", "nom_prospect", "medecin", "médecin"]))
    write_cell(ws, "C3", parse_excel_date(first_value(data, ["date_visite", "date", "date_rapport", "date_visite_medicale", "visite.date"])))
    write_cell(ws, "C4", first_value(data, ["objectif_visite", "objectif", "type_visite"]))

    for row in range(7, 13):
        for col in range(1, 6):
            ws.cell(row, col).value = None

    products, extra_products = normalize_products(data)

    for idx, product in enumerate(products, start=7):
        write_cell(ws, f"A{idx}", product.get("produit"))
        write_cell(ws, f"B{idx}", product.get("commentaire"))
        write_cell(ws, f"C{idx}", product.get("opportunites"))
        write_cell(ws, f"D{idx}", product.get("benchmarking"))
        write_cell(ws, f"E{idx}", product.get("echantillons"))

    write_cell(ws, "B14", first_value(data, ["nom_superviseur", "superviseur", "supervisor"] ))
    write_cell(ws, "D14", first_value(data, ["nombre_patients_presents", "nombre_patients", "patients_presents", "patients_présents"] ))
    write_cell(ws, "F14", list_to_text(first_value(data, ["gadgets", "gadget", "supports", "outils"] )))

    remarque = first_value(data, ["remarque_generale", "remarque_générale", "resume_rapport", "commentaire_visite", "rapport_final"])
    if extra_products:
        extra_names = ", ".join(clean_cell_value(item.get("produit")) for item in extra_products if clean_cell_value(item.get("produit")))
        if extra_names:
            remarque = f"{clean_cell_value(remarque) or ''}\nProduits supplémentaires: {extra_names}".strip()

    write_cell(ws, "B15", remarque)
    write_cell(ws, "B17", parse_excel_date(first_value(data, ["date_relance", "date_prochaine_visite", "prochaine_date", "relance.date"])))
    write_cell(ws, "D17", first_value(data, ["prochaine_action", "prochaine_etape", "prochaine_étape", "next_step"] ))

    for cell in ["C3", "B17"]:
        ws[cell].number_format = "dd/mm/yyyy"

    update_medecin_sheet(wb, data)
    refill_report_validation(wb)

    return wb


def create_template_xlsx_from_json(json_input, output_xlsx_path=None, template_path=None):
    if isinstance(json_input, (str, Path)):
        json_path = Path(json_input)
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        default_stem = json_path.stem.replace("_project_pi_groq", "")
    elif isinstance(json_input, dict):
        data = json_input
        default_stem = Path(str(data.get("file_name", "rapport"))).stem
    else:
        raise TypeError("json_input doit être un chemin JSON ou un dictionnaire Python.")

    template = resolve_template_path(template_path)
    wb = load_workbook(template)
    fill_template_workbook(wb, data)

    OUTPUT_TEMPLATE_XLSX_DIR.mkdir(parents=True, exist_ok=True)

    if output_xlsx_path is None:
        safe_stem = re.sub(r"[^A-Za-z0-9À-ÿ_.-]+", "_", default_stem).strip("_") or "rapport"
        output_xlsx_path = OUTPUT_TEMPLATE_XLSX_DIR / f"{safe_stem}_rapport_template.xlsx"
    else:
        output_xlsx_path = Path(output_xlsx_path)
        output_xlsx_path.parent.mkdir(parents=True, exist_ok=True)

    wb.save(output_xlsx_path)
    return output_xlsx_path


def export_all_json_to_template_xlsx(json_dir=OUTPUT_JSON_DIR, output_dir=OUTPUT_TEMPLATE_XLSX_DIR, template_path=None):
    json_dir = Path(json_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    json_files = sorted(json_dir.glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"Aucun JSON trouvé dans {json_dir}")

    generated_files = []
    for json_file in json_files:
        output_xlsx = output_dir / f"{json_file.stem.replace('_project_pi_groq', '')}_rapport_template.xlsx"
        generated_files.append(create_template_xlsx_from_json(json_file, output_xlsx, template_path))

    print(f"✅ {len(generated_files)} fichier(s) XLSX généré(s) dans: {output_dir}")
    return generated_files


generated_xlsx_files = export_all_json_to_template_xlsx()
generated_xlsx_files[:5]

